# 第二部分 2.2：无监督学习

| 章节 | 内容 |
|------|------|
| **2.3 降维** | PCA 原理、SVD vs PCA |
| **2.4 聚类算法** | K-Means、DBSCAN |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification, make_regression, make_blobs, make_moons
from sklearn.model_selection import train_test_split

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print('环境就绪')

## 2.3 降维

高维数据直接使用会带来「维度灾难」：计算量爆炸、样本稀疏、可视化困难。降维的目标是**用更少的维度保留最多的信息**。

### PCA（Principal Component Analysis，主成分分析）

PCA 是最经典的无监督降维方法，将高维数据投影到**方差最大**的方向上。

**核心思路**：
1. **中心化**：减去列均值，消除背景偏移（必须做）
2. 计算协方差矩阵，找出方差最大的方向（主成分）
3. 按解释方差比从大到小排列，取前 k 个方向投影

**典型用途**：
- 高维数据可视化（压缩到 2D/3D 画图）
- 消除冗余特征，减少计算量
- 去除噪声（低方差方向往往是噪声）

**主成分数量怎么选**？看**累计解释方差比**，通常选覆盖 80%~95% 方差所需的最少主成分数。

### SVD vs PCA

第一章的 SVD 与 PCA 密切相关：

| | 操作 | 适用场景 |
|--|------|---------|
| **原始 SVD** | 直接分解矩阵 A | 矩阵本身有意义（如词频矩阵） |
| **PCA** | 先中心化，再做 SVD | 统计降维（特征的方差有意义） |

`sklearn.PCA` 内部就是对**中心化后**的数据矩阵做 SVD，两者数学等价。特征量纲不同时（如身高+体重+年龄）才需要额外加 `StandardScaler` 再做 PCA。

下面用第一章的 20 篇文章 × 10 词汇数据，直观对比两者的差异。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ── 与第一章完全相同的文章词频数据 ───────────────────────────
words = ["模型","训练","数据","神经网络","文本","图像","词向量","奖励","策略","分类"]
articles = [
    "监督学习入门","决策树原理","随机森林调参","梯度提升方法","正则化技术",
    "词向量技术",  "语言模型综述","文本分类方法","机器翻译进展","情感分析实践",
    "图像识别综述","卷积神经网络","目标检测算法","图像分割技术","视觉Transformer",
    "强化学习基础","策略梯度方法","深度Q网络",  "多智能体系统","游戏AI应用",
]
topic_labels = ['机器学习']*5 + ['自然语言处理']*5 + ['计算机视觉']*5 + ['强化学习']*5
colors       = ['C0']*5 + ['C1']*5 + ['C2']*5 + ['C3']*5

base = np.array([
    [8, 6, 7, 2, 1, 1, 0, 0, 0, 5],
    [4, 3, 3, 2, 8, 1, 7, 0, 0, 6],
    [3, 5, 4, 7, 1, 9, 0, 0, 0, 4],
    [2, 4, 2, 5, 0, 0, 0, 8, 9, 2],
], dtype=float)
np.random.seed(42)
A = np.vstack([base[i // 5] + np.random.randint(0, 3, size=10) for i in range(20)])

# ── 核心区别：PCA 先对数据做中心化 ───────────────────────────
col_mean   = A.mean(axis=0)    # 每个词在所有文章的平均词频 (10,)
A_centered = A - col_mean      # 减均值后：体现"比平均多/少用了哪些词"

print("每个词的平均词频（PCA 要先减去这部分背景值）：")
for w, m in zip(words, col_mean):
    print(f"  {w:<5} 均值 = {m:.1f}")

# ── 方法一：SVD 直接作用于原始矩阵（第一章做法）────────────
_, _, Vt_raw = np.linalg.svd(A, full_matrices=False)
coords_raw   = A @ Vt_raw[:2].T            # shape: (20, 2)

# ── 方法二：手动中心化后做 SVD（= PCA 的数学本质）───────────
_, _, Vt_cen = np.linalg.svd(A_centered, full_matrices=False)
coords_cen   = A_centered @ Vt_cen[:2].T  # shape: (20, 2)

# ── 方法三：sklearn PCA（内部自动中心化 + SVD）───────────────
pca2        = PCA(n_components=2)
coords_pca  = pca2.fit_transform(A)        # 与方法二数学等价（方向可能镜像）

# ── 验证等价性 ────────────────────────────────────────────────
print(f"\nsklearn PCA 与手动中心化 SVD 坐标对比（前 3 篇文章）：")
print(f"  {'文章':<12} {'手动中心化 SVD':>22} {'sklearn PCA':>22}")
for i in range(3):
    a = f"({coords_cen[i,0]:+.2f}, {coords_cen[i,1]:+.2f})"
    b = f"({coords_pca[i,0]:+.2f}, {coords_pca[i,1]:+.2f})"
    print(f"  {articles[i]:<12} {a:>22} {b:>22}")
print("  → 数值相同（轴方向可能镜像，但空间结构完全一致）")

# ── 可视化：三图对比 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

def scatter_articles(ax, coords, title, xlabel='', ylabel=''):
    seen = set()
    for i in range(20):
        t = topic_labels[i]
        ax.scatter(coords[i, 0], coords[i, 1], color=colors[i], s=90,
                   label=t if t not in seen else None, zorder=3)
        seen.add(t)
        ax.annotate(articles[i], (coords[i, 0], coords[i, 1]),
                    fontsize=6.5, xytext=(4, 2), textcoords='offset points')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.legend(fontsize=8, loc='lower right')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

scatter_articles(axes[0], coords_raw,
    '原始 SVD（第一章方法）\n第一方向含均值偏移（词频整体高低）')
scatter_articles(axes[1], coords_cen,
    'PCA = 中心化后做 SVD\n第一方向 = 偏离均值最大的方向（纯主题差异）')

# 右图：scree plot（PCA 解释方差比）
pca_full = PCA(n_components=10).fit(A)
expl     = pca_full.explained_variance_ratio_
cumul    = np.cumsum(expl)
axes[2].bar(range(1, 11), expl * 100, color='steelblue', alpha=0.8)
axes[2].plot(range(1, 11), cumul * 100, 'ro-', ms=5, label='累计占比')
axes[2].axhline(90, color='gray', linestyle='--', alpha=0.7, label='90% 阈值')
axes[2].set_xlabel('主成分编号')
axes[2].set_ylabel('解释方差比 (%)')
axes[2].set_title(f'PCA 解释方差比（Scree Plot）\n前 2 个主成分包含 {cumul[1]:.0%} 的方差信息')
axes[2].legend(fontsize=8)
axes[2].set_xticks(range(1, 11))
for i, v in enumerate(expl[:4]):
    axes[2].text(i + 1, v * 100 + 0.5, f'{v:.0%}', ha='center', fontsize=8)

plt.suptitle('SVD vs PCA：唯一的区别是 PCA 先对数据做中心化（减去列均值）', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

# ── 总结：为什么有 SVD 还需要 PCA？ ──────────────────────────
print("\n=== 为什么有了 SVD 还需要 PCA？ ===\n")
print("  SVD（原始矩阵）:")
print("    第一奇异方向由词频绝对量主导，含均值偏移（文章整体词频高低）")
print("    适合：矩阵结构分解，如推荐系统的用户-商品矩阵\n")
print("  PCA = 先减均值，再做 SVD:")
print("    消除「背景词频」，只保留「偏差」：这篇文章比平均多用了哪些词？")
print("    第一主成分 = 区分文章主题差异最显著的方向")
print("    适合：数据分析、可视化、特征降维\n")
print("  关系总结:")
print("    np.linalg.svd(A)              → 纯数学工具，直接分解")
print("    sklearn.PCA.fit_transform(A)  → 自动中心化 + SVD + 解释方差比\n")
print("  何时还需要 StandardScaler（除以标准差）？")
print("    特征量纲不同时：身高(cm)、体重(kg)、年龄(岁) 方差差异巨大")
print("    需要 StandardScaler → PCA，否则方差大的特征会主导结果")
print("    本例词频量纲一致（都是次数），只中心化即可，无需 StandardScaler")

## 2.4 聚类算法

聚类是无监督学习——没有标签，目标是找到数据内部的**自然分组**。

### K-Means

**算法步骤**：
1. 随机初始化 k 个聚类中心（质心）
2. 将每个样本分配到最近的质心（硬分配）
3. 更新质心为该簇所有点的均值
4. 重复 2-3 直到质心不再移动（收敛）

**如何选 k？** 两种常用方法配合使用：

| 方法 | 思路 | 选法 |
|------|------|------|
| **肘部法（Elbow Method）** | 画出不同 k 对应的簇内误差（Inertia），即所有点到各自质心距离的平方和 | 选下降趋势明显变缓的「肘部」值 |
| **轮廓系数（Silhouette Score）** | 衡量每个点与同簇的紧密度 vs 与最近邻簇的分离度，范围 -1~1，越高越好 | 选轮廓系数最大的 k |

### K-Means++ 初始化优化

普通 K-Means 随机选择初始质心，容易陷入**局部最优**（不同初始化得到差异很大的结果），且需要更多迭代。

**K-Means++ 策略**：
1. 随机选第一个质心
2. 后续质心按**与已选质心的距离平方**为概率选取——距离越远的点被选为下一个质心的概率越高
3. 初始质心因此「分散」在数据各处，接近真实聚类中心

**效果**：更低的 Inertia、更少的迭代次数、更稳定的结果。sklearn 默认 `init='k-means++'`。

| 初始化方式 | Inertia 稳定性 | 收敛速度 | sklearn 参数 |
|------------|---------------|---------|-------------|
| 随机（random） | 波动大，易局部最优 | 慢 | `init='random'` |
| **K-Means++** | 稳定，结果更优 | 快 | `init='k-means++'`（默认） |

**局限**：假设簇是**球形**的，需要预先指定 k，无法识别任意形状的簇，也不能自动发现异常点。

In [ ]:

# ── 2.4 KMeans 实战：露营领域词汇聚类 ────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from modelscope import snapshot_download
from sentence_transformers import SentenceTransformer

# ── 词汇列表（露营主题，共 157 个）────────────────────────────
words = [
    # 自然地形
    "山野","河谷","湖畔","溪边","山顶","林地","草坪","荒原","山谷","草原",
    "森林","旷野","郊野","秘境",
    # 营地类型
    "湖边营地","森林营地","海岛露营","房车营地","木屋营地","河滩空地",
    "林间空地","原生山林","森林公园","野外空地","高原地带","深山","丛林",
    "浅滩","避风湾","天然草坪",
    # 露营装备
    "帐篷","天幕","睡袋","防潮垫","露营车","收纳箱","营地灯","头灯",
    "焚火台","卡式炉","气罐","煮水壶","套锅","天幕杆","地钉","风绳",
    "登山鞋","冲锋衣","登山包","急救包","工兵铲","驱蚊液","保温箱",
    "吊床","挡风板","炉头","木炭","防水袋","地布","户外餐具","折叠桌椅",
    # 户外活动
    "徒步","登山","溯溪","骑行","越野","攀岩","飞盘","野外垂钓",
    "丛林穿越","野外探险","短途出游","荒野漫游","登高望远","山谷徒步",
    "森林漫步","短途自驾","山野旅居","野外生存","溪涧玩水","野外踏青",
    "春日野餐","高原徒步","雪地露营","夜间徒步","轻户外","精致露营",
    "极简露营","山野放空",
    # 野外饮食
    "烧烤","烤肉","火锅","野外煮茶","手冲咖啡","冰镇饮料","野餐糕点",
    "三明治","柴火煮饭","野外煲汤","速冻食材","烧烤酱料","便携速食",
    "气泡水","果酒","林间下午茶","山野美食","锡纸烧烤","铁板烤肉",
    "罐头食品","坚果干果","时令水果","篝火聚餐",
    # 自然景象
    "星空","晚霞","日出","云海","晚风","晨雾","落日","月光","星河",
    "山间清风","林间光影","山野暮色","漫天星光","落日余晖","旷野晚风",
    "远山云雾","林间晨光","深山夜色","山野清风","林间落叶","碧空万里",
    "静谧山林","静水湖面","破晓晨光",
    # 心境与生活方式
    "松弛感","慢生活","解压放松","自愈放空","逃离喧嚣","沉浸式露营",
    "独处时光","亲子露营","好友结伴","小众旅行","自然治愈","亲近自然",
    "回归山野","城市出逃","休闲度假","户外康养","山野氛围感","户外社交",
    "周末放松","山野疗愈","自由随性",
]
print(f"词汇总数：{len(words)}")

# ── 从 ModelScope 加载模型 ────────────────────────────────────
print("\n正在加载模型...")
model_dir       = snapshot_download('Jerry0/text2vec-base-chinese')
encoder         = SentenceTransformer(model_dir)
embeddings      = encoder.encode(words, show_progress_bar=True)
embeddings_norm = normalize(embeddings)
print(f"\n向量维度：{embeddings.shape}  →  {embeddings.shape[1]} 维语义向量")

# ── 肘部法确定最优 k ──────────────────────────────────────────
inertias = []
k_range  = range(2, 16)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    km.fit(embeddings_norm)
    inertias.append(km.inertia_)

# 二阶差分定量找肘部：下降量本身下降最快的位置
delta2  = np.diff(np.diff(np.array(inertias)))
elbow_k = list(k_range)[np.argmax(delta2) + 1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), inertias, 'bo-', ms=6, lw=1.5)
ax.axvline(elbow_k, color='red', linestyle='--', lw=1.5,
           label=f'肘部 k={elbow_k}（二阶差分法）')
for k, v in zip(k_range, inertias):
    ax.text(k, v + 0.3, str(k), ha='center', fontsize=7.5, color='gray')
ax.set_xlabel('k（聚类数）')
ax.set_ylabel('Inertia（簇内误差平方和）')
ax.set_title('肘部法（Elbow Method）\n寻找 Inertia 下降明显趋缓的拐点')
ax.legend()
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.show()

print(f"\n→ 肘部法选定 k = {elbow_k}")

# ── 训练最终 KMeans ───────────────────────────────────────────
optimal_k = elbow_k
km_final  = KMeans(n_clusters=optimal_k, init='k-means++', n_init=30, random_state=42)
labels    = km_final.fit_predict(embeddings_norm)

# ── PCA 降维到 2D 可视化 ──────────────────────────────────────
pca           = PCA(n_components=2, random_state=42)
coords        = pca.fit_transform(embeddings_norm)
var_explained = pca.explained_variance_ratio_.sum()

palette        = matplotlib.colormaps['tab10'].resampled(optimal_k)
cluster_colors = [palette(i) for i in range(optimal_k)]

fig, (ax_scatter, ax_bar) = plt.subplots(1, 2, figsize=(15, 6))

for i in range(optimal_k):
    mask = labels == i
    ax_scatter.scatter(coords[mask, 0], coords[mask, 1],
                       color=cluster_colors[i], s=40, alpha=0.85,
                       label=f'簇 {i}', zorder=3)
    for j in np.where(mask)[0]:
        ax_scatter.annotate(words[j], (coords[j, 0], coords[j, 1]),
                            fontsize=5.5, alpha=0.75,
                            xytext=(2, 2), textcoords='offset points')

centers_2d = pca.transform(km_final.cluster_centers_)
ax_scatter.scatter(centers_2d[:, 0], centers_2d[:, 1],
                   marker='*', s=200, c='black', zorder=5, label='簇中心')
ax_scatter.set_title(f'KMeans 聚类结果（k={optimal_k}）\nPCA 2D 投影（解释方差 {var_explained:.1%}）')
ax_scatter.legend(fontsize=7, loc='upper right')
ax_scatter.set_xlabel(f'PC1（{pca.explained_variance_ratio_[0]:.1%}）')
ax_scatter.set_ylabel(f'PC2（{pca.explained_variance_ratio_[1]:.1%}）')

cluster_sizes = [(labels == i).sum() for i in range(optimal_k)]
bars = ax_bar.bar(range(optimal_k), cluster_sizes,
                  color=cluster_colors, edgecolor='white', linewidth=0.5)
for bar, size in zip(bars, cluster_sizes):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(size), ha='center', fontsize=9)
ax_bar.set_xlabel('簇编号')
ax_bar.set_ylabel('词汇数量')
ax_bar.set_title('各簇词汇数量分布')
ax_bar.set_xticks(range(optimal_k))
ax_bar.set_xticklabels([f'簇 {i}' for i in range(optimal_k)])

plt.suptitle('露营词汇 KMeans 聚类分析', fontsize=12)
plt.tight_layout()
plt.show()

# ── 输出各簇词汇 ──────────────────────────────────────────────
print("\n" + "="*60)
print(f"  各簇词汇明细（k={optimal_k}）")
print("="*60)
for i in range(optimal_k):
    cluster_words = [words[j] for j in range(len(words)) if labels[j] == i]
    print(f"\n【簇 {i}】（{len(cluster_words)} 个词）")
    for row_start in range(0, len(cluster_words), 8):
        print("  " + "、".join(cluster_words[row_start:row_start+8]))


### DBSCAN

基于**密度**：密度足够高的区域形成一个簇，稀疏区域的点被标记为**噪声点（异常点）**。

**两个核心参数**：
- `eps`：邻域半径，定义「足够近」的范围
- `min_samples`：在 eps 半径内至少需要多少邻居，才能成为核心点

**三类点的角色**：

| 点类型 | 判断条件 | 说明 |
|--------|----------|------|
| **核心点** | eps 邻域内邻居数 ≥ min_samples | 密集区域的内部点 |
| **边界点** | 邻居数 < min_samples，但在某核心点邻域内 | 簇的边缘点 |
| **噪声点（-1）** | 不是核心点，也不在任何核心点邻域内 | 离群异常点 |

**K-Means vs DBSCAN 对比**：

| | K-Means | DBSCAN |
|--|---------|--------|
| 需要预设簇数量 | ✓ 必须指定 k | ✗ 自动确定 |
| 簇的形状假设 | 球形 | **任意形状** |
| 噪声/异常点检测 | ✗ | ✓ 自动标记为 -1 |
| 适合场景 | 球形、均匀大小的簇 | 环形、月牙形、不规则形状 |

**参数选择经验**：
- `eps` 过大 → 所有点合并为一个大簇；过小 → 几乎所有点都是噪声
- `min_samples` 通常设为特征维度 × 2，或根据数据密度调整
- 实用技巧：对每个点计算第 k 近邻距离，排序后画折线图，**折线拐点对应合适的 eps**

In [ ]:

# ── 2.4 DBSCAN 实战：环形数据聚类对比 ────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_circles
from sklearn.neighbors import NearestNeighbors

np.random.seed(42)

# ── 生成环形数据 + 随机噪声点 ─────────────────────────────────
X_rings, y_rings = make_circles(n_samples=400, noise=0.05, factor=0.4, random_state=42)
noise_pts = np.random.uniform(-1.5, 1.5, size=(25, 2))
X = np.vstack([X_rings, noise_pts])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── 左图：原始数据 ────────────────────────────────────────────
axes[0].scatter(X_rings[:, 0], X_rings[:, 1], c=y_rings, cmap='RdBu',
                s=15, alpha=0.85, label='环形样本')
axes[0].scatter(noise_pts[:, 0], noise_pts[:, 1],
                c='gray', s=30, marker='x', linewidths=1.2, label='噪声点')
axes[0].set_title('原始数据\n内外两个环形簇 + 随机噪声点', fontsize=10)
axes[0].legend(fontsize=8)
axes[0].set_aspect('equal')
axes[0].set_xlim(-1.7, 1.7)
axes[0].set_ylim(-1.7, 1.7)

# ── 中图：K-Means（失败）────────────────────────────────────
km = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
km_labels = km.fit_predict(X)

axes[1].scatter(X[:, 0], X[:, 1], c=km_labels, cmap='RdBu', s=15, alpha=0.85)
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                marker='*', s=250, c='black', zorder=5, label='质心')
axes[1].set_title('K-Means（k=2）[失败]\n以距离为准，按左右劈开，无法识别环形', fontsize=10)
axes[1].legend(fontsize=8)
axes[1].set_aspect('equal')
axes[1].set_xlim(-1.7, 1.7)
axes[1].set_ylim(-1.7, 1.7)

# ── 右图：DBSCAN（成功）──────────────────────────────────────
dbscan = DBSCAN(eps=0.15, min_samples=5)
db_labels = dbscan.fit_predict(X)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()

palette = ['C0', 'C1', 'C2', 'C3']
for label in sorted(set(db_labels)):
    mask = db_labels == label
    if label == -1:
        axes[2].scatter(X[mask, 0], X[mask, 1], c='gray', s=40,
                        marker='x', linewidths=1.2, zorder=4,
                        label=f'噪声点（-1）：{mask.sum()} 个')
    else:
        axes[2].scatter(X[mask, 0], X[mask, 1], c=palette[label % len(palette)],
                        s=15, alpha=0.85, label=f'簇 {label}：{mask.sum()} 个点')

axes[2].set_title(f'DBSCAN（eps=0.15，min_samples=5）[成功]\n发现 {n_clusters} 个环形簇，识别 {n_noise} 个噪声点', fontsize=10)
axes[2].legend(fontsize=8, loc='upper right')
axes[2].set_aspect('equal')
axes[2].set_xlim(-1.7, 1.7)
axes[2].set_ylim(-1.7, 1.7)

plt.suptitle('环形数据聚类：K-Means（失败）vs DBSCAN（成功）\nK-Means 假设球形簇，DBSCAN 基于密度、可识别任意形状',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# ── 附：k 近邻距离图（eps 参数选择技巧）────────────────────
k = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(X)
distances, _ = nbrs.kneighbors(X)
kth_dist = np.sort(distances[:, -1])[::-1]

fig2, ax2 = plt.subplots(figsize=(8, 3))
ax2.plot(kth_dist, lw=1.5)
ax2.axhline(0.15, color='red', linestyle='--', lw=1.5, label='eps=0.15（选定值）')
ax2.set_xlabel('点的索引（按距离降序排列）')
ax2.set_ylabel(f'第 {k} 近邻距离')
ax2.set_title(f'k 近邻距离图（k={k}）：折线拐点对应合适的 eps 值')
ax2.legend()
plt.tight_layout()
plt.show()

print("DBSCAN 结果摘要：")
print(f"  eps=0.15，min_samples=5")
print(f"  发现簇数量：{n_clusters}")
print(f"  噪声点数量：{n_noise}（占 {n_noise/len(X):.1%}）")
for label in sorted(set(db_labels)):
    if label != -1:
        print(f"  簇 {label}：{(db_labels == label).sum()} 个点")
